<a href="https://colab.research.google.com/github/soham-never-codes/Festiva-AI-Event-Planner/blob/main/notebooks/Festiva_Week3_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#  Mount Drive (always first)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/festiva'

os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/rag', exist_ok=True)

print("✅ Drive mounted. Project folder architecture ready.")

Mounted at /content/drive
✅ Drive mounted. Project folder architecture ready.


In [2]:
!pip install -q transformers torch

In [3]:
#  Zero-shot event type classifier ──────────────────────────────
from transformers import pipeline

print("🧠 Loading BART-Large-MNLI to the GPU... (Takes ~1-2 mins to download)")

# Using zero-shot — no labelled training data needed at all!
# device=0 forces the model to use Colab's T4 GPU for lightning-fast inference
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)
print("✅ NLP Classifier loaded and active on GPU!")

🧠 Loading BART-Large-MNLI to the GPU... (Takes ~1-2 mins to download)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ NLP Classifier loaded and active on GPU!


In [4]:
# Entity extractor & Classifier ───────────────────────────────
import re

CITIES = ["bangalore", "bengaluru", "mumbai", "delhi", "chennai",
          "hyderabad", "pune", "kolkata", "jaipur", "goa"]

CITY_NORM = {"bengaluru": "Bangalore"}

def extract_entities(text):
    entities = {}

    # Budget: handles ₹10L, 10 lakhs, ₹5,00,000 etc.
    patterns = [
        (r'₹\s*(\d+(?:\.\d+)?)\s*(?:L\b|lakh|lakhs)', 1e5),
        (r'(\d+(?:\.\d+)?)\s*(?:L\b|lakh|lakhs)', 1e5),
        (r'₹\s*([\d,]+)', 1),
    ]

    for pat, mult in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            val = float(m.group(1).replace(',','')) * mult
            # [FIXED] Changed from 'budget' to 'total_budget' to match Week 2 model
            entities['total_budget'] = int(val)
            break

    # City extraction
    for city in CITIES:
        if city in text.lower():
            entities['city'] = CITY_NORM.get(city, city.capitalize())
            break

    # Guest count extraction
    m = re.search(r'(\d+)\s+(?:guests?|people|persons?|attendees?)', text, re.IGNORECASE)
    if m:
        entities['guest_count'] = int(m.group(1))

    return entities

def classify_event(text):
    # These labels are slightly longer to help the AI understand the context better
    labels = ["wedding", "corporate event", "birthday party", "engagement", "conference"]
    res = classifier(text, labels)
    raw = res['labels'][0]

    # Normalize label back to our exact Week 2 categories
    clean = raw.replace(' party','').replace(' event','').replace(' ','_')

    return {'event_type': clean, 'confidence': round(res['scores'][0], 3)}

In [5]:
#  Test with sample queries ─────────────────────────────────────
test_queries = [
    "Wedding in Bangalore, budget ₹10L, around 200 guests",
    "Corporate conference in Mumbai for 100 employees, ₹5 lakhs",
    "My daughter's birthday party in Pune, 50 kids, budget ₹80000",
    "Engagement ceremony in Delhi with 300 guests, ₹15L budget",
    "Tech summit in Hyderabad, 500 attendees, ₹20 lakhs"
]

print(f"{'Query':<60} | {'Event':<12} | {'City':<10} | {'Budget (₹)':>12} | {'Guests':>8}")
print("-" * 115)

for q in test_queries:
    clf = classify_event(q)
    ent = extract_entities(q)

    # Truncate query for clean printing
    short_q = q[:57] + "..." if len(q) > 60 else q

    print(f"{short_q:<60} | {clf['event_type']:<12} | "
          f"{ent.get('city','—'):<10} | "
          f"₹{ent.get('total_budget',0):>11,} | "
          f"{ent.get('guest_count','—'):>8}")

Query                                                        | Event        | City       |   Budget (₹) |   Guests
-------------------------------------------------------------------------------------------------------------------
Wedding in Bangalore, budget ₹10L, around 200 guests         | wedding      | Bangalore  | ₹  1,000,000 |      200
Corporate conference in Mumbai for 100 employees, ₹5 lakhs   | corporate    | Mumbai     | ₹    500,000 |        —
My daughter's birthday party in Pune, 50 kids, budget ₹80000 | birthday     | Pune       | ₹     80,000 |        —
Engagement ceremony in Delhi with 300 guests, ₹15L budget    | engagement   | Delhi      | ₹  1,500,000 |      300
Tech summit in Hyderabad, 500 attendees, ₹20 lakhs           | conference   | Hyderabad  | ₹  2,000,000 |      500


In [6]:
#  Save classifier config to Drive ──────────────────────────────
import joblib

# We save the lists and names so our FastAPI app knows exactly what to load
nlp_config = {
    'cities': CITIES,
    'city_norm': CITY_NORM,
    'model_name': 'facebook/bart-large-mnli',
    'labels': ["wedding", "corporate event", "birthday party", "engagement", "conference"]
}

joblib.dump(nlp_config, f'{PROJECT_DIR}/models/nlp_config.pkl')
print("✅ NLP Configuration successfully saved to Drive!")

✅ NLP Configuration successfully saved to Drive!
